# STEP 3 — Routing Sweep (conf_gate, entropy_gate, step_gate, cascade_3lvl)

**Prerequisite:** Steps 0–2 complete. Results from Step 2 attached via slm-agents-results.

**What it does:**
- Sweeps all threshold values for conf_gate, entropy_gate, step_gate across both datasets
- Runs the 3-level cascade with a grid of (t1, t2) pairs
- Checkpoints after every (policy, threshold, slm, dataset) combo

**Outputs:** `results/routing_episodes.csv` — all routing sweep results

**Estimated run time:** ~8 hours  
**IMPORTANT:** Run session_save at the end and upload results.

In [ ]:
import sys, gc, json
import torch
import pandas as pd
from pathlib import Path

sys.path.insert(0, '/kaggle/working/slm_agents_paper')
from src.session_init import init
P = init()

from src.models   import load_model, get_model_by_name
from src.datasets import load_samples
from src.routing  import (
    ConfidenceGateRouter, EntropyGateRouter, StepCountRouter, ThreeLevelCascadeRouter,
    CONF_THRESHOLDS, ENTROPY_THRESHOLDS, STEP_KS, CASCADE_GRID
)
from src.react_env import run_react_episode, episode_to_row
from src.metrics   import compute_summary

hotpotqa = load_samples(P['data'] / 'hotpotqa.json')
musique  = load_samples(P['data'] / 'musique.json')
all_data = {'hotpotqa': hotpotqa, 'musique': musique}
print('Data loaded')

In [ ]:
# Resume-safe
ROUT_PATH = P['results'] / 'routing_episodes.csv'
if ROUT_PATH.exists():
    rout_df = pd.read_csv(ROUT_PATH)
    done_keys = set()
    for (policy, slm, ds), grp in rout_df.groupby(['policy','slm_name','dataset']):
        if len(grp) >= len(all_data[ds]):
            done_keys.add((policy, slm, ds))
    print(f'⚡ Resuming — {len(done_keys)} combos done')
else:
    rout_df = pd.DataFrame()
    done_keys = set()

all_rows = rout_df.to_dict('records') if not rout_df.empty else []

In [ ]:
# Keep LLM resident — swap only SLM per condition to save time
slm_1b_cfg = get_model_by_name('Qwen2.5-1.5B')
llm_cfg    = get_model_by_name('Llama-3.1-8B')

llm_model, llm_tok = load_model(llm_cfg)
print('LLM loaded and resident')

In [ ]:
def run_policy_sweep(policy_id, router, slm_cfg, slm_model, slm_tok):
    """Run one routing policy across both datasets, appending to all_rows."""
    for ds_name, samples in all_data.items():
        key = (policy_id, slm_cfg.name, ds_name)
        if key in done_keys:
            print(f'  ✓ Skip {key}')
            return
        print(f'  [{policy_id}] {ds_name} ({len(samples)} q)...')
        for i, sample in enumerate(samples):
            ep = run_react_episode(
                sample, slm_model, slm_tok, slm_cfg,
                llm_model, llm_tok, llm_cfg,
                router, policy_id,
            )
            all_rows.append(episode_to_row(ep))
            if (i + 1) % 20 == 0:
                pd.DataFrame(all_rows).to_csv(ROUT_PATH, index=False)
        pd.DataFrame(all_rows).to_csv(ROUT_PATH, index=False)
        done_keys.add(key)
        sr = sum(r['success'] for r in all_rows if r['policy']==policy_id and r['dataset']==ds_name)
        n  = sum(1           for r in all_rows if r['policy']==policy_id and r['dataset']==ds_name)
        llm_r = sum(r['llm_call_ratio'] for r in all_rows if r['policy']==policy_id and r['dataset']==ds_name) / n
        print(f'  → SR={sr/n:.3f}, LLM_ratio={llm_r:.3f}')

In [ ]:
# Load SLM-1.5B and sweep conf_gate + entropy_gate + step_gate
slm_model, slm_tok = load_model(slm_1b_cfg)
print('\n=== SLM: Qwen2.5-1.5B ===')

print('\n-- conf_gate sweep --')
for T in CONF_THRESHOLDS:
    run_policy_sweep(f'conf_gate_T{T:.1f}', ConfidenceGateRouter(T), slm_1b_cfg, slm_model, slm_tok)

print('\n-- entropy_gate sweep --')
for T in ENTROPY_THRESHOLDS:
    run_policy_sweep(f'entropy_gate_T{T:.1f}', EntropyGateRouter(T), slm_1b_cfg, slm_model, slm_tok)

print('\n-- step_gate sweep --')
for K in STEP_KS:
    run_policy_sweep(f'step_gate_K{K}', StepCountRouter(K), slm_1b_cfg, slm_model, slm_tok)

del slm_model, slm_tok
gc.collect(); torch.cuda.empty_cache()
print('SLM-1.5B unloaded')

In [ ]:
# 3-level cascade: needs both 1.5B (l1) and 3B (l2) loaded
slm_1b_m, slm_1b_t = load_model(slm_1b_cfg)
slm_3b_cfg = get_model_by_name('Qwen2.5-3B')
slm_3b_m, slm_3b_t = load_model(slm_3b_cfg)
print('\n=== 3-Level Cascade sweep ===')

# The cascade logic runs two rounds of generate_single_with_logits
# We implement it inline here because it needs two SLMs
from src.inference import generate_single_with_logits
from src.react_env import build_react_prompt, parse_react_action, _execute_search, _is_finish_action, _extract_finish_answer, _exact_match, EpisodeResult, StepResult, episode_to_row
from src.datasets import build_bm25_index
import re

for t1, t2 in CASCADE_GRID:
    router = ThreeLevelCascadeRouter(t1, t2)
    policy_id = f'cascade_t1{t1:.1f}_t2{t2:.1f}'
    for ds_name, samples in all_data.items():
        key = (policy_id, 'cascade_1b_3b_8b', ds_name)
        if key in done_keys:
            continue
        print(f'  [{policy_id}] {ds_name}...')
        for sample in samples:
            question = sample['question']
            gold     = sample['answer']
            passages = sample['passages']
            bm25     = build_bm25_index(passages)
            scratchpad = ''
            n_llm = n_slm = 0
            predicted = ''
            for step_idx in range(8):
                prompt = build_react_prompt(question, scratchpad)
                # Level 1: 1.5B
                out1 = generate_single_with_logits(slm_1b_m, slm_1b_t, prompt)
                if router.decide_level1(out1['first_token_entropy']):
                    # Level 2: 3B
                    out2 = generate_single_with_logits(slm_3b_m, slm_3b_t, prompt)
                    if router.decide_level2(out2['first_token_entropy']):
                        # Level 3: 8B
                        final_out = generate_single_with_logits(llm_model, llm_tok, prompt)
                        n_llm += 1
                    else:
                        final_out = out2; n_slm += 1
                else:
                    final_out = out1; n_slm += 1
                resp    = final_out['response']
                parsed  = parse_react_action(resp)
                is_term = _is_finish_action(parsed['action'], parsed['action_input'], resp)
                if is_term:
                    predicted = _extract_finish_answer(parsed['action'], parsed['action_input'], resp)
                    break
                obs = _execute_search(parsed['action_input'] or parsed['thought'], bm25, passages)
                scratchpad += f"\nThought: {parsed['thought']}\nAction: {parsed['action']}[{parsed['action_input']}]\nObservation: {obs}\n"
            success = _exact_match(predicted or '', gold)
            n_total = n_llm + n_slm
            all_rows.append({
                'question_id': sample['id'], 'dataset': ds_name,
                'policy': policy_id, 'slm_name': 'cascade_1b_3b_8b',
                'success': success, 'predicted': predicted, 'gold_answer': gold,
                'n_steps': step_idx+1, 'n_llm_calls': n_llm, 'n_slm_calls': n_slm,
                'llm_call_ratio': n_llm/n_total if n_total>0 else 0,
            })
        pd.DataFrame(all_rows).to_csv(ROUT_PATH, index=False)
        done_keys.add(key)
        sr = sum(r['success'] for r in all_rows if r['policy']==policy_id and r['dataset']==ds_name)
        n  = sum(1           for r in all_rows if r['policy']==policy_id and r['dataset']==ds_name)
        print(f'  → SR={sr/n:.3f}')

del slm_1b_m, slm_1b_t, slm_3b_m, slm_3b_t
gc.collect(); torch.cuda.empty_cache()
print('\n✅ STEP 3 COMPLETE')

In [ ]:
from src.session_save import save
save(P['results'])
print('⚠️  Upload results before closing session')